# 🇰🇷 Hello Clova Agent — HyperCLOVA X SEED Text-Instruct 1.5B

**한국어 프롬프트 한 줄 → Reveal.js 발표 슬라이드 자동 생성**  
**국산 LLM 경량 버전 | T4 Free Tier 최적화 | 양자화 없이 즉시 실행**

---

## Think-14B 대비 개선 사항

| 항목 | Think-14B (`HCX14B_Colab`) | Text-Instruct 1.5B (이 노트북) |
|------|:--------------------------:|:------------------------------:|
| 모델 크기 | 28 GB (4-bit ≈ 9 GB) | **3 GB** |
| Colab 에러 수 | 3가지 | **1가지** |
| 모델 로딩 | 20~40분 | **~1-2분** |
| 양자화 | bitsandbytes 4-bit 필수 | **불필요** |
| transformers 버전 | >=5.9.0 필수 | **표준 버전 OK** |
| bitsandbytes 설치 | 필요 | **불필요** |

> **남은 에러 1가지**: `libcudart.so.13` — vLLM CUDA ABI 문제 (모델 크기와 무관).  
> `apt-get install cuda-cudart-13-0` 한 줄로 해결 → 셀 1에서 자동 처리됩니다.

---

## 실행 방법

**셀을 위에서 아래로 순서대로 실행하세요.** (단축키: `Shift + Enter`)

> ⏱️ 최초 실행 시 모델 다운로드 + 서버 준비 약 **2~5분** 소요됩니다.  
> Colab 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 필수

---

## 사전 준비 (필수 — 셀 실행 전 완료)

> ⚠️ **naver-hyperclovax 모델은 HuggingFace gated repo입니다.**

**1단계 — 모델 접근 신청** (1회, 즉시 승인)  
아래 페이지에서 "Agree and access repository" 클릭:  
→ https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B

**2단계 — HF_TOKEN Colab Secrets 등록** (세션마다)  
Colab 왼쪽 패널 🔑 **Secrets** → `HF_TOKEN` 키로 토큰 추가 → "노트북 접근 허용" 토글 ON  
토큰 발급: https://huggingface.co/settings/tokens (Read 권한 이상)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 1/5  ▌ 환경 진단 + CUDA 13 런타임 설치 + vLLM 서버 시작
#
# [Think-14B 대비 제거된 것]
#   ✗ pip install transformers>=5.9.0  (표준 버전으로 동작)
#   ✗ pip install bitsandbytes         (양자화 없음)
#   ✗ --quantization bitsandbytes
#   ✗ --load-format bitsandbytes
#   ✗ --enforce-eager                  (CUDA Graph 정상 동작)
#   ✗ --trust-remote-code              (표준 아키텍처)
#
# [유지되는 것]
#   ✅ cuda-cudart-13-0 설치   (vLLM ABI 문제는 모델 크기와 무관)
#   ✅ --dtype half            (T4 Compute 7.5 = float16만 지원)
# ══════════════════════════════════════════════════════════════════════

import subprocess, os, sys, torch

HCX_MODEL = 'naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B'

# ── 1) GPU / CUDA 환경 진단 ───────────────────────────────────────────
print('🔍 CUDA 환경 진단 중...')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader \
    2>/dev/null || echo '  ⚠️ GPU 없음 — 런타임 → T4 GPU 선택 후 재실행하세요'
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.version.cuda or "unknown"}')

# ── 2) HuggingFace 로그인 (필수) ─────────────────────────────────────
# naver-hyperclovax 모델은 gated repo입니다. 셀 실행 전 사전 준비 완료 필수:
#   1) HuggingFace 접근 신청 (모델 페이지 → Agree and access)
#   2) Colab 🔑 Secrets → HF_TOKEN 등록
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN = ''
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN 미설정\n'
        '1) https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B 에서 접근 신청\n'
        '2) Colab 왼쪽 🔑 Secrets → HF_TOKEN 추가 → 세션 재시작'
    )
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print(f'  ✅ HuggingFace 로그인 완료 (token: {HF_TOKEN[:8]}...)')

# ── 3) CUDA 13 런타임 설치 ───────────────────────────────────────────
# vLLM PyPI 휠(Python 3.12)은 CUDA 13 기준 컴파일됩니다.
# Colab T4 환경(CUDA 12.x)과 ELF VERNEED 태그가 불일치 → 실제 패키지 설치 필요.
# 심볼릭 링크로는 파일명만 해결되고 버전 태그는 해결 불가 — 이 단계가 핵심입니다.
print('\n🔧 CUDA 13 런타임 설치 중... (약 10초)')
result = subprocess.run(
    ['apt-get', 'install', '-y', '-q', 'cuda-cudart-13-0'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('❌ CUDA 13 런타임 설치 실패:')
    print(result.stderr[-500:])
    raise RuntimeError('cuda-cudart-13-0 설치 실패')
print('  ✅ cuda-cudart-13-0 설치 완료')

cuda13_lib = '/usr/local/cuda-13.0/lib64'
ld = os.environ.get('LD_LIBRARY_PATH', '')
if cuda13_lib not in ld:
    os.environ['LD_LIBRARY_PATH'] = f'{cuda13_lib}:{ld}'
subprocess.run(['ldconfig'], capture_output=True)
print(f'  🔗 LD_LIBRARY_PATH += {cuda13_lib}')

# ── 4) vLLM 설치 ─────────────────────────────────────────────────────
# 1.5B는 양자화 없이 fp16으로 직접 로드 → bitsandbytes 불필요
# 표준 transformers 버전으로 동작 (transformers>=5.9.0 별도 설치 불필요)
print('\n📦 vLLM 설치 중... (약 2-3분)')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'vllm', '-q'])
print('  ✅ vLLM 설치 완료')

# ── 5) vLLM import 사전 검증 ─────────────────────────────────────────
print('\n🔍 vLLM import 검증 중...')
env = os.environ.copy()
check = subprocess.run(
    [sys.executable, '-c',
     'from vllm.entrypoints.openai import api_server; print("OK")'],
    capture_output=True, text=True, timeout=60, env=env,
)
if check.returncode != 0:
    print('❌ vLLM import 실패 — 서버 시작을 중단합니다.')
    for line in (check.stderr or check.stdout).strip().splitlines()[-10:]:
        print(f'   {line}')
    raise RuntimeError('vLLM import 실패')
print('  ✅ vLLM import 검증 통과')

# ── 6) vLLM 서버 백그라운드 시작 ────────────────────────────────────
print(f'\n🚀 vLLM API 서버 시작: {HCX_MODEL}')
print('   fp16 직접 로드 (양자화 없음) — VRAM ~3 GB 사용')
print('   모델 로딩 ~1-2분 소요 (셀 2~3 실행하면서 대기)')

vllm_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model',                  HCX_MODEL,
        '--host',                   '0.0.0.0',
        '--port',                   '8000',
        '--dtype',                  'half',      # T4 Compute 7.5 = float16만 지원. bfloat16은 Ampere(8.0+)부터.
        '--max-model-len',          '4096',
        '--gpu-memory-utilization', '0.85',
        '--served-model-name',      HCX_MODEL,
    ],
    env=env,
    stdout=open('/tmp/vllm.log', 'w'),
    stderr=subprocess.STDOUT,
)

os.environ['VLLM_PID'] = str(vllm_proc.pid)
print(f'\n✅ vLLM 프로세스 시작 (PID: {vllm_proc.pid})')
print('   → 셀 2, 3 실행 후 셀 4에서 준비 완료 대기')
print('   → 로그 확인: !tail -f /tmp/vllm.log')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 2/5  ▌ 프로젝트 코드 준비 (git clone / git pull)
#
# 처음 실행: 저장소 전체를 내려받습니다 (git clone)
# 재실행 시: 최신 코드로 업데이트합니다  (git pull)
# ══════════════════════════════════════════════════════════════════════

import os

REPO_URL = 'https://github.com/machine-artisan/Hello-Clova-Agent.git'
REPO_DIR = 'Hello-Clova-Agent'

if os.path.exists(REPO_DIR):
    print('🔄 최신 코드로 업데이트 중...')
    !git -C {REPO_DIR} pull
else:
    print('📂 프로젝트 다운로드 중...')
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f'\n✅ 현재 경로: {os.getcwd()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 3/5  ▌ 의존성 설치 + 환경변수 설정
#
# [pip 충돌 경고 안내]
#   vLLM이 starlette/opentelemetry 버전을 변경하면서
#   Colab 사전설치 패키지(google-adk 등)와 충돌 경고가 발생합니다.
#   → 이 프로젝트(langgraph, gradio, openai)와 무관 — 무시해도 됩니다.
# ══════════════════════════════════════════════════════════════════════

import os, sys, subprocess, importlib

# ── 1) 프로젝트 의존성 설치 ──────────────────────────────────────────
print('📦 프로젝트 의존성 설치 중 (requirements.txt)...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)

IGNORE_PATTERNS = [
    'google-adk', 'prometheus-fastapi', 'opentelemetry', 'starlette',
    'dependency resolver', 'behaviour is the source', 'does not currently take',
]
real_errors = [
    line for line in (result.stdout + result.stderr).splitlines()
    if ('error' in line.lower() or 'failed' in line.lower())
    and not any(pat in line.lower() for pat in IGNORE_PATTERNS)
    and line.strip()
]
if real_errors:
    print('❌ 프로젝트 패키지 설치 실패:')
    for e in real_errors:
        print(f'   {e}')
else:
    print('   ✅ requirements.txt 설치 완료')
    print('   (Colab 자체 패키지 충돌 경고는 이 프로젝트와 무관 — 무시)')

# ── 2) 핵심 패키지 import 검증 ──────────────────────────────────────
print('\n🔍 핵심 패키지 검증:')
all_ok = True
for pkg_name, import_name in [
    ('langgraph', 'langgraph'), ('openai', 'openai'),
    ('gradio', 'gradio'), ('python-dotenv', 'dotenv'),
]:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, '__version__', '설치됨')
        print(f'   ✅ {pkg_name} ({ver})')
    except ImportError as e:
        print(f'   ❌ {pkg_name}: {e}')
        all_ok = False

if not all_ok:
    raise RuntimeError('핵심 패키지 import 실패 — 위 오류를 확인하세요')

# ── 3) 환경변수 설정 ─────────────────────────────────────────────────
HCX_MODEL = 'naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B'
os.environ['LLM_API_BASE'] = 'http://localhost:8000/v1'
os.environ['LLM_API_KEY']  = 'EMPTY'
os.environ['LLM_MODEL']    = HCX_MODEL

print(f'\n✅ 환경변수 설정 완료')
print(f'   LLM_API_BASE = {os.environ["LLM_API_BASE"]}')
print(f'   LLM_MODEL    = {os.environ["LLM_MODEL"]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 4/5  ▌ vLLM 서버 준비 완료 대기
#
# 1.5B 모델은 통상 1~2분 내 준비됩니다 (14B: 20~40분).
# 프로세스 종료 감지 시 즉시 전체 로그를 출력합니다.
# ══════════════════════════════════════════════════════════════════════

import time, urllib.request, subprocess, os

HEALTH_URL = 'http://localhost:8000/health'
MAX_WAIT   = 300   # 5분 (14B는 1200초/20분)
INTERVAL   = 10
LOG_EVERY  = 60

vllm_pid = int(os.environ.get('VLLM_PID', '0'))

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except (ProcessLookupError, PermissionError):
        return False

def show_log(n=30, label=''):
    result = subprocess.run(['tail', f'-{n}', '/tmp/vllm.log'],
                            capture_output=True, text=True)
    if label:
        print(label)
    print(result.stdout or '  (로그 없음)')

def show_engine_error():
    result = subprocess.run(['grep', '-A', '8', 'EngineCore.*ERROR', '/tmp/vllm.log'],
                            capture_output=True, text=True)
    if result.stdout.strip():
        print('── EngineCore 에러 (실제 원인) ──')
        print(result.stdout)

print(f'⏳ vLLM 서버 준비 대기 중 (최대 {MAX_WAIT//60}분, PID: {vllm_pid})\n')

elapsed, next_log_at, server_ready = 0, LOG_EVERY, False

while elapsed < MAX_WAIT:
    if vllm_pid and not pid_alive(vllm_pid):
        print(f'\n❌ vLLM 프로세스(PID {vllm_pid})가 {elapsed}s에 비정상 종료')
        show_engine_error()
        show_log(n=60, label='── /tmp/vllm.log 최근 60줄 ──')
        break

    try:
        urllib.request.urlopen(HEALTH_URL, timeout=3)
        server_ready = True
        break
    except Exception:
        pass

    print(f'  [{elapsed:3d}s] 대기 중...', end='\r')
    if elapsed >= next_log_at:
        show_log(n=20, label=f'\n── [{elapsed}s] /tmp/vllm.log 최근 20줄 ──')
        next_log_at += LOG_EVERY

    time.sleep(INTERVAL)
    elapsed += INTERVAL

if server_ready:
    print(f'\n✅ vLLM 서버 준비 완료! ({elapsed//60}분 {elapsed%60}초)')
    print('   → 셀 5를 실행해 에이전트를 시작하세요')
elif not (vllm_pid and not pid_alive(vllm_pid)):
    print(f'\n❌ {MAX_WAIT//60}분 내 서버 시작 실패.')
    show_engine_error()
    show_log(n=60, label='── 전체 로그 (최근 60줄) ──')

## 🏗️ 시스템 아키텍처

```
┌─────────────────────────────────────────────────────────┐
│                   브라우저 (여러분의 PC)                 │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (공개 URL)
┌───────────────────────▼─────────────────────────────────┐
│         🌐 웹서버  —  Gradio  (포트 7860)               │
└───────────────────────┬─────────────────────────────────┘
                        │  Python 함수 호출
┌───────────────────────▼─────────────────────────────────┐
│         ⚙️  앱서버  —  LangGraph 파이프라인             │
│   Node1 input_parser → Node2 outline_generator          │
│   Node3 slide_writer → Node4 html_renderer              │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (OpenAI API 형식)
┌───────────────────────▼─────────────────────────────────┐
│         🤖 API서버  —  vLLM  (포트 8000)                │
└───────────────────────┬─────────────────────────────────┘
                        │  GPU fp16 (양자화 없음)
┌───────────────────────▼─────────────────────────────────┐
│   🇰🇷 LLM — HyperCLOVA X SEED Text-Instruct 1.5B      │
│         VRAM ~3 GB  |  T4 15 GB에서 여유롭게 동작       │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 5/5  ▌ 에이전트 실행 🚀
#
# 셀 실행 후 출력되는 공개 URL을 브라우저에서 열어주세요.
#   예) Running on public URL: https://xxxx.gradio.live
#
# 중지하려면: 셀 왼쪽의 ■ 버튼 클릭
# ══════════════════════════════════════════════════════════════════════

print('🚀 Hello Clova Agent (HyperCLOVA X SEED Text-Instruct 1.5B) 시작 중...')
print('   아래에 공개 URL이 출력되면 브라우저에서 접속하세요 👇')
print()

!python ui/app.py

---

## 🔬 Phase 2 선택 실행 — 임베딩 모델 (RAG 준비)

아래 셀은 **Phase 2 RAG 연동**을 위한 임베딩 모델 설정입니다.  
Phase 1 체험만 원하면 실행하지 않아도 됩니다.

| 항목 | 내용 |
|------|------|
| 모델 | `BAAI/bge-m3` (다국어 임베딩, 한국어 우수) |
| 용도 | 외부 문서를 벡터로 변환하여 RAG 검색에 활용 |
| 크기 | ~570 MB |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [선택] Phase 2  ▌ 임베딩 모델 준비 (RAG용)
# ══════════════════════════════════════════════════════════════════════

import os

print('📦 임베딩 관련 패키지 설치 중...')
!pip install langchain-huggingface sentence-transformers -q

from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'BAAI/bge-m3'

if not os.path.exists(EMBED_MODEL):
    print(f'📥 임베딩 모델 다운로드 중: {EMBED_MODEL}')
    model = SentenceTransformer(EMBED_MODEL)
    model.save(EMBED_MODEL)
    print('✅ 다운로드 완료 — 로컬 캐시에 저장됨')
else:
    print(f'✅ 캐시된 모델 사용: {EMBED_MODEL}')

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda', 'trust_remote_code': True},
    encode_kwargs={'normalize_embeddings': True},
)

test_vec = embeddings.embed_query('안녕하세요, 임베딩 테스트입니다.')
print(f'\n✅ 임베딩 모델 준비 완료')
print(f'   벡터 차원: {len(test_vec)}')
print(f'   샘플 벡터 (앞 5개): {[round(v, 4) for v in test_vec[:5]]}')

---

## 📚 더 알아보기

### 코드 구조

```
Hello-Clova-Agent/
├── agent/
│   ├── state.py              ← 파이프라인이 공유하는 데이터 구조
│   ├── llm.py                ← vLLM API 클라이언트 (OpenAI 호환)
│   ├── graph.py              ← LangGraph 파이프라인 조립
│   └── nodes/
│       ├── input_parser.py   ← Node 1: 입력 분석
│       ├── outline_generator.py  ← Node 2: 목차 생성 (LLM)
│       ├── slide_writer.py   ← Node 3: 내용 작성 (LLM)
│       └── html_renderer.py  ← Node 4: HTML 변환
└── ui/app.py                 ← Gradio 웹 UI
```

### HyperCLOVA X SEED 모델 라인업

| 모델 | VRAM | 에러 수 | 비고 |
|------|------|---------|------|
| **Text-Instruct-1.5B** ← 이 노트북 | ~3 GB | **1가지** | T4 권장 |
| Think-14B (`HCX14B_Colab`) | ~9 GB (4-bit) | 3가지 | Colab Pro 환경 권장 |
| Think-32B | ~18 GB (4-bit) | — | A100 전용 |

---
*Hello Clova Agent · Phase 1 MVP · LangGraph + vLLM + HyperCLOVA X SEED Text-Instruct 1.5B + Reveal.js*